# How to solve stochastic programs with SPAROW

**This notebook contains a demonstration for solving a simple stochastic programming exemplar with SPAROW**

In [1]:
### Import the newsvendor exemplar from sparow
'''
    If solving your own model, see https://github.com/sandialabs/sparow_examples/ for examples of structuring the application data, 
    scenario data, Pyomo model builder(s), and stochastic programming model object (including specifying a bundling scheme).
'''
from sparow.sp.examples import simple_newsvendor


[    0.00] Initializing mpi-sppy
Alternative solutions package from or_topas is available.


## Solving optimization problems with the Extensive Form

In [2]:
### Import EF solver
from sparow.ef.ef import ExtensiveFormSolver

### Import pprint for solution readability
import pprint


In [3]:
example = simple_newsvendor()   # newsvendor example imported from sparow.sp.examples 
solver = ExtensiveFormSolver()  # solving with extensive form
solver.set_options(
    solver="gurobi",   # solving with gurobi
    loglevel="INFO",   # can replace with DEBUG, VERBOSE, etc.
)
results = solver.solve(example.sp)  # example.sp is the model object; this line returns the solution object
pprint.pprint(results.to_dict())    # pretty-prints results


INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
/home/rmalfan/projects/forest/sparow/sparow/sp/bundling/bundling_helper_functions.py:123: UserWarning: No scenario probabilities are given; assuming uniform distribution.
  warnings.warn(
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP


{'metadata': {'as_solution_source': 'sparow.solnpool.solnpool._sparow_as_solution',
              'context_name': None,
              'end_time': '2026-07-30 19:22:39.925312',
              'policy': 'keep_best',
              'start_time': '2026-07-30 19:22:39.868611',
              'status': 'ok',
              'termination_condition': 'optimal',
              'time_elapsed': '0:00:00.056701'},
 'pool_config': {'abs_tolerance': 0.0,
                 'best_value': 76.5,
                 'max_pool_size': None,
                 'objective': 0,
                 'rel_tolerance': None,
                 'sense_is_min': True},
 'solutions': {0: {'id': 0,
                   'objectives': [{'index': None,
                                   'name': None,
                                   'suffix': {},
                                   'value': 76.5}],
                   'suffix': {},
                   'variables': [{'discrete': False,
                                  'fixed': False,
       

ExtensiveFormSolver utilizes the single_bundle scenario bundling scheme (i.e., all scenarios in a single subproblem) to return a Sparow solution pool containing metadata, pool_config, and solutions.

metadata subdictionary:
| Key | Value |
| --- | --- |
| solution time | TODO: I think this is the wall clock time but double check |
| status | Solver status (see the [Pyomo docs](https://www.pyomo.org/blog/2015/1/8/accessing-solver)) |
| termination condition | Termination condition (see the [Pyomo docs](https://www.pyomo.org/blog/2015/1/8/accessing-solver)) |
| context_name | Name of the solution pool; this will be None for EF solves because only a single pool exists |
| as_solution_source | Name of the function standardizing the solution pool |
| policy | Pool Policy; this will be "keep_best" for any EF solve |

pool_config subdictionary:
| Key | Value |
| --- | --- |
| abs_tolerance | Absolute gap tolerance |
| rel_tolerance | Relative gap tolerance |
| best_value | "Best" objective value; this will be the optimal objective value for any EF solve |
| max_pool_size | Largest number of solutions that can be stored in the pool; this will be None for any EF solve because the best solution is the only one stored by default |
| objective | Index of solution pool containing the objective; this will be 0 for any EF solve because there is only 1 pool stored |
| sense_is_min | True if the user specifies a minimization optimization model, False if maximization |

The keys of the solutions subdictionary are the indices associated with each solution in the pool. Each solution index has a dictionary with keys and values:
| Key | Value |
| --- | --- |
| id | TODO: how is this different from the solution index? |
| objectives | TODO: are index, name, suffix ever populated? 'value' corresponds to the optimal objective value |
| suffix | This will be empty for any EF solve |
| variables | 'value' is the value of the optimal solution for variable 'name' at 'index' |

## Solving optimization problems with Serial PH

In [4]:
### Import serial PH solver
from sparow.ph.ph import ProgressiveHedgingSolver


In [5]:
example = simple_newsvendor()           # newsvendor example imported from sparow.sp.examples 
solver = ProgressiveHedgingSolver()     # solving with serial PH
solver.set_options(
    solver="gurobi",   # solving with gurobi
    max_iterations=2,  # this will default to 100
    loglevel="INFO",   # can replace with DEBUG, VERBOSE, etc.
    rho_updates=True,  # rho parameter will update at each iteration
)

results = solver.solve(example.sp)  # example.sp is the model object; this line returns the solution object
pprint.pprint(results.to_dict())    # pretty-prints results


INFO - ProgressiveHedgingSolver - START
WARNING - Variable objective coefficient is 0; rho0 set to 1.5
INFO - 
INFO - ----------------------------------------------------------------------
INFO - Iteration:        0
INFO - obj_lb:           61.400000000000006
INFO - conv_norm:        None
INFO - xbar_diff_norm:   None
INFO - time:             2026-07-30 19:22:40.003211
INFO - time_last_iter:   0.06741928914561868
INFO - 
WARNING - Variable objective coefficient is 0; rho0 set to 1.5
INFO - g = 18.60800000400537
INFO - G = 0.0933333316355558
INFO - 
INFO - ----------------------------------------------------------------------
INFO - Iteration:        1
INFO - obj_lb:           -353.29933331388776
INFO - conv_norm:        18.60800000400537
INFO - xbar_diff_norm:   0.0933333316355558
INFO - time:             2026-07-30 19:22:40.070433
INFO - time_last_iter:   0.06450174003839493
INFO - 
WARNING - Variable objective coefficient is 0; rho0 set to 1.5
INFO - g = 1.8780815480567983e-08
INFO -

{'metadata': {'as_solution_source': 'sparow.solnpool.solnpool._sparow_as_solution',
              'context_name': 'Finalized Last PH Solution',
              'policy': 'keep_best'},
 'pool_config': {'abs_tolerance': 0.0,
                 'best_value': 76.66986666772829,
                 'max_pool_size': None,
                 'objective': 0,
                 'rel_tolerance': None,
                 'sense_is_min': True},
 'solutions': {3: {'id': 3,
                   'objectives': [{'index': None,
                                   'name': None,
                                   'suffix': {},
                                   'value': 76.66986666772829}],
                   'suffix': {'g': 1.8780815480567983e-08,
                              'iteration': 2,
                              'obj_lb': 76.67640002458113},
                   'variables': [{'discrete': False,
                                  'fixed': False,
                                  'index': 0,
                     

For PH solves:

1. 'policy' within the 'metadata' subdictionary has options: keep_all, keep_best, keep_latest, keep_latest_unique, keep_pareto.

2. TODO: will 'context' within the 'metadata' subdictionary always be 'Finalized Last PH Solution', or will it change depending on 'policy'? 

3. The 'suffix' subdictionary within 'solutions' will contain:

| Key | Value |
| --- | --- |
| g | Variable value that is computed to determine whether or not the termination condition has been met |
| iteration | The iteration number for which the solution was obtained |
| obj_lb | The lower bound on the optimal objective value computed at the given iteration |

4. The 'suffix' subdictionaries within the 'variables' subdictionary within 'solutions' will contain 'w': the weights associated with each subproblem.

## Solving optimization problems with Parallel PH

Run the command under "%%bash" in the command line if not running in a Jupyter notebook.
- See "driver_mpi.py" for an example of a driver for executing the mpi-sppy wrapper for solving the facility location exemplar.
- "-np" refers to the number of processors and is a user-specified flag.

In [6]:
%%bash 
mpiexec -np 4 python driver_mpi.py


[    0.00] Initializing mpi-sppy
Alternative solutions package from or_topas is available.
Alternative solutions package from or_topas is available.
Alternative solutions package from or_topas is available.
Alternative solutions package from or_topas is available.


INFO - ProgressiveHedgingSolver_MPISPPY - START


[    0.03] Initializing SPBase
[    0.05] Initializing PHBase
[    0.06] Starting spcomm.main()
[    0.06] Creating solvers
[    0.07] Entering solve loop in PHBase.Iter0

After PH Iteration 0
Trivial bound = 16762583.791071428
PHBase Convergence Metric = None
Elapsed time:   0.18
[    0.22] Iter.           Best Bound  Best Incumbent      Rel. Gap        Abs. Gap
[    0.22]     0  *     16762583.7911             inf           inf%             inf
[    0.22] Initiating PH Iteration 1

[    0.36]     1  L     16762584.0972             inf           inf%             inf

After PH Iteration 1
Scaled PHBase Convergence Metric= 0.13605442176870755
Iteration time:   0.14
Elapsed time:     0.33
[    0.36] Initiating PH Iteration 2

[    0.49]     2  L X   16762584.4033   16772469.5054         0.059%       9885.1020
[    0.49] Terminating based on inter-cylinder relative gap        0.059%
[    0.49] Cylinder convergence

Invoking scenario reporting functions, if applicable


Invoking PH extensi

INFO - 
INFO:sparow.logs:
INFO - ----------------------------------------------------------------------
INFO:sparow.logs:----------------------------------------------------------------------
INFO - ProgressiveHedgingSolver_MPISPPY - FINALIZING
INFO:sparow.logs:ProgressiveHedgingSolver_MPISPPY - FINALIZING
INFO - 
INFO:sparow.logs:
INFO - ----------------------------------------------------------------------
INFO:sparow.logs:----------------------------------------------------------------------
INFO - ProgressiveHedgingSolver_MPISPPY - STOP
INFO:sparow.logs:ProgressiveHedgingSolver_MPISPPY - STOP


{'metadata': {'as_solution_source': 'sparow.solnpool.solnpool._sparow_as_solution',
              'context_name': 'PH Iterations',
              'end_time': '2026-07-30 19:22:43.160433',
              'lower_bound': 16762584.403316326,
              'policy': 'keep_latest',
              'solver': 'PH Iteration Results',
              'solver_options': {'convergence_tolerance': 0.001,
                                 'max_iterations': 2,
                                 'normalize_convergence_norm': True,
                                 'solver_name': 'gurobi',
                                 'solver_options': {},
                                 'time_limit': None},
              'start_time': '2026-07-30 19:22:42.590101',
              'termination_condition': 'ok',
              'time_elapsed': '0:00:00.570332'},
 'pool_config': {'max_pool_size': 1},
 'solutions': {2: {'id': 2,
                   'objectives': [{'index': None,
                                   'name': None,
     